# FHOPS Onboarding — Playback & KPIs

This notebook demonstrates schedule playback and KPI computation
using the FHOPS evaluation layer.

## Playback pipeline

1. **Load scenario** — `fhops.scenario.io.load_scenario` + `Problem.from_scenario`.
2. **Run playback** — `fhops.evaluation.playback.run_playback(pb, assignments)` converts
   raw machine/block/day/shift assignments into shift-level records.
3. **Aggregate** — `shift_dataframe()` and `day_dataframe()` collapse records into
   per-shift and per-day summary DataFrames with production, mobilisation cost,
   utilisation, blackout conflicts, and sequencing violations.
4. **Compute KPIs** — `fhops.evaluation.compute_kpis(pb, assignments)` returns a
   `KPIResult` with scalar totals plus attached shift/day calendars.

## What the KPIs measure

| Category | Key metrics |
| --- | --- |
| Production | `total_production`, `staged_production`, `remaining_work_total`, `completed_blocks`, `makespan_day` |
| Mobilisation | `mobilisation_cost`, `mobilisation_cost_by_machine`, `mobilisation_cost_by_landing` |
| Utilisation | `utilisation_ratio_mean_shift`, `utilisation_ratio_weighted_shift`, `utilisation_ratio_mean_day` |
| Sequencing | `sequencing_violation_count`, `sequencing_violation_blocks`, `sequencing_violation_days` |

## Budget

- `tiny7` and `small21` solve with 50 and 100 SA iterations respectively.
- All output is computed in-memory; no files are written to disk.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "examples"))
from notebook_support import find_repo_root, discover_scenarios, run_fhops_cli
from fhops.scenario.io import load_scenario
from fhops.scenario.contract import Problem
from fhops.optimization.heuristics import solve_sa
from fhops.evaluation import (
    run_playback,
    compute_kpis,
    shift_dataframe,
    day_dataframe,
    PlaybackConfig,
)
import pandas as pd

repo_root = find_repo_root()
print(f'Repo root: {repo_root}')

Repo root: /srv/shared-data/gep/jupyterhub04-projects/fhops


In [2]:
# Solve tiny7 and small21
schedules = {}
for kind, subpath in [('tiny7', 'tiny7'), ('small21', 'small21')]:
    sc = load_scenario(str(repo_root / 'examples' / subpath / 'scenario.yaml'))
    pb = Problem.from_scenario(sc)
    iters = 50 if kind == 'tiny7' else 100
    print(f'Solving {kind} with {iters} iterations...', flush=True)
    res = solve_sa(pb, iters=iters, seed=7)
    schedules[kind] = {'scenario': sc, 'problem': pb, 'assignments': res['assignments']}
    print(f'  objective={res["objective"]:.3f}  rows={len(res["assignments"])}')
print('Done.')
display(pd.DataFrame([{
    'scenario': k, 'objective': v['assignments'].shape[0], 'blocks': len(v['scenario'].blocks)
} for k, v in schedules.items()]).round(2))

Solving tiny7 with 50 iterations...
  objective=4306.523  rows=23
Solving small21 with 100 iterations...
  objective=15574.425  rows=90
Done.


,scenario,objective,blocks
0,tiny7,23,2
1,small21,90,6


## Run deterministic playback

`run_playback(pb, assignments)` produces `PlaybackResult` with shift-level
records and aggregated summaries.

In [3]:
for kind in ['tiny7', 'small21']:
    s = schedules[kind]
    result = run_playback(s['problem'], s['assignments'])
    print(f'{kind}: {len(result.records)} records, '
          f'{len(result.shift_summaries)} shift summaries, '
          f'{len(result.day_summaries)} day summaries')
    print(f'  delivered_total={result.delivered_total:.1f}  '
          f'remaining_work={result.remaining_work_total:.1f}')

tiny7: 23 records, 23 shift summaries, 7 day summaries
  delivered_total=4414.7  remaining_work=0.0
small21: 90 records, 90 shift summaries, 21 day summaries
  delivered_total=15967.0  remaining_work=0.0


## Shift-level summary DataFrame

Columns: `day`, `shift_id`, `machine_id`, `machine_role`, `production_units`,
`total_hours`, `idle_hours`, `mobilisation_cost`, `sequencing_violations`,
`blackout_conflicts`, `available_hours`, `utilisation_ratio`.

In [4]:
for kind in ['tiny7', 'small21']:
    s = schedules[kind]
    result = run_playback(s['problem'], s['assignments'])
    shift_df = shift_dataframe(result)
    print(f'\n=== {kind} shift summary ({len(shift_df)} rows) ===')
    display(shift_df.head(15))


=== tiny7 shift summary (23 rows) ===


,day,shift_id,machine_id,machine_role,sample_id,production_units,total_hours,idle_hours,mobilisation_cost,sequencing_violations,blackout_conflicts,available_hours,utilisation_ratio,downtime_hours,downtime_events,weather_severity_total
0,1,S1,H1,feller_buncher,0,1220.906942,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
1,1,S1,H2,feller_buncher,0,1185.883055,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
2,2,S1,H1,feller_buncher,0,1194.708925,24.0,0.0,53.24,0,0,24.0,1.0,0.0,0,0.0
3,2,S1,H2,feller_buncher,0,813.203830,24.0,0.0,53.24,0,0,24.0,1.0,0.0,0,0.0
4,2,S1,H3,grapple_skidder,0,1332.369187,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
5,3,S1,H3,grapple_skidder,0,1480.641393,24.0,0.0,53.24,0,0,24.0,1.0,0.0,0,0.0
6,3,S1,H4,processor,0,531.895933,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
7,3,S1,H5,processor,0,531.895933,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
8,4,S1,H3,grapple_skidder,0,527.271362,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
9,4,S1,H4,processor,0,569.826226,24.0,0.0,53.24,0,0,24.0,1.0,0.0,0,0.0



=== small21 shift summary (90 rows) ===


,day,shift_id,machine_id,machine_role,sample_id,production_units,total_hours,idle_hours,mobilisation_cost,sequencing_violations,blackout_conflicts,available_hours,utilisation_ratio,downtime_hours,downtime_events,weather_severity_total
0,1,S1,H1,feller_buncher,0,1176.844281,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
1,1,S1,H2,feller_buncher,0,1220.906942,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
2,2,S1,H1,feller_buncher,0,787.363583,24.0,0.0,53.24,0,0,24.0,1.0,0.0,0,0.0
3,2,S1,H2,feller_buncher,0,1220.906942,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
4,2,S1,H3,grapple_skidder,0,1164.728216,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
5,3,S1,H1,feller_buncher,0,1176.844281,24.0,0.0,53.24,0,0,24.0,1.0,0.0,0,0.0
6,3,S1,H2,feller_buncher,0,1220.906942,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
7,3,S1,H3,grapple_skidder,0,1332.369187,24.0,0.0,53.48,0,0,24.0,1.0,0.0,0,0.0
8,3,S1,H4,processor,0,517.165642,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0
9,3,S1,H5,processor,0,517.165642,24.0,0.0,0.00,0,0,24.0,1.0,0.0,0,0.0


## Day-level summary DataFrame

Columns: `day`, `production_units`, `total_hours`, `idle_hours`,
`mobilisation_cost`, `completed_blocks`, `sequencing_violations`,
`utilisation_ratio`.

In [5]:
for kind in ['tiny7', 'small21']:
    s = schedules[kind]
    result = run_playback(s['problem'], s['assignments'])
    day_df = day_dataframe(result)
    print(f'\n=== {kind} day summary ({len(day_df)} rows) ===')
    display(day_df)


=== tiny7 day summary (7 rows) ===


,day,sample_id,production_units,total_hours,idle_hours,mobilisation_cost,completed_blocks,blackout_conflicts,sequencing_violations,available_hours,utilisation_ratio,downtime_hours,downtime_events,weather_severity_total
0,1,0,2406.789997,48.0,168.0,0.00,0,0,0,216.0,0.222222,0.0,0,0.0
1,2,0,3340.281942,72.0,144.0,106.48,0,0,0,216.0,0.333333,0.0,0,0.0
2,3,0,2544.433259,72.0,144.0,53.24,0,0,0,216.0,0.333333,0.0,0,0.0
3,4,0,2469.335168,96.0,120.0,106.48,0,0,0,216.0,0.444444,0.0,0,0.0
4,5,0,2879.865938,96.0,120.0,106.48,0,0,0,216.0,0.444444,0.0,0,0.0
5,6,0,2413.726061,120.0,96.0,106.48,1,0,0,216.0,0.555556,0.0,0,0.0
6,7,0,1604.378643,48.0,168.0,106.48,1,0,0,216.0,0.222222,0.0,0,0.0



=== small21 day summary (21 rows) ===


,day,sample_id,production_units,total_hours,idle_hours,mobilisation_cost,completed_blocks,blackout_conflicts,sequencing_violations,available_hours,utilisation_ratio,downtime_hours,downtime_events,weather_severity_total
0,1,0,2397.751223,48.0,168.0,0.00,0,0,0,216.0,0.222222,0.0,0,0.0
1,2,0,3172.998741,72.0,144.0,53.24,0,0,0,216.0,0.333333,0.0,0,0.0
2,3,0,4764.451694,120.0,96.0,106.72,0,0,0,216.0,0.555556,0.0,0,0.0
3,4,0,4882.664567,144.0,72.0,160.20,0,0,0,216.0,0.666667,0.0,0,0.0
4,5,0,5095.808491,168.0,48.0,160.44,0,0,0,216.0,0.777778,0.0,0,0.0
5,6,0,6097.319213,168.0,48.0,53.48,0,0,0,216.0,0.777778,0.0,0,0.0
6,7,0,5408.517603,168.0,48.0,160.68,0,0,0,216.0,0.777778,0.0,0,0.0
7,8,0,5777.605298,192.0,24.0,214.40,1,0,0,216.0,0.888889,0.0,0,0.0
8,9,0,4421.983917,168.0,48.0,214.40,0,0,0,216.0,0.777778,0.0,0,0.0
9,10,0,4333.607766,144.0,72.0,106.72,0,0,0,216.0,0.666667,0.0,0,0.0


## Compute KPIs

`compute_kpis(pb, assignments)` returns a `KPIResult` with scalar totals
plus attached shift/day calendars.

In [6]:
kpis = {}
for kind in ['tiny7', 'small21']:
    s = schedules[kind]
    kpis[kind] = compute_kpis(s['problem'], s['assignments'])

rows = []
for kind in ['tiny7', 'small21']:
    k = kpis[kind]
    rows.append({
        'scenario': kind,
        'total_production': k['total_production'],
        'completed_blocks': k['completed_blocks'],
        'mobilisation_cost': k['mobilisation_cost'],
        'utilisation_ratio_mean_shift': k['utilisation_ratio_mean_shift'],
        'utilisation_ratio_mean_day': k['utilisation_ratio_mean_day'],
        'sequencing_violation_count': k['sequencing_violation_count'],
        'makespan_day': k['makespan_day'],
    })
pd.DataFrame(rows).round(3)

,scenario,total_production,completed_blocks,mobilisation_cost,utilisation_ratio_mean_shift,utilisation_ratio_mean_day,sequencing_violation_count,makespan_day
0,tiny7,4414.703,2.0,585.64,1.0,0.365,0,7
1,small21,15966.967,6.0,2564.40,1.0,0.476,0,17


## Mobilisation cost breakdown

Per-machine and per-landing mobilisation costs help identify which
machines or landings drive relocation spend.

In [7]:
for kind in ['tiny7', 'small21']:
    k = kpis[kind]
    print(f'\n=== {kind} mobilisation cost ===')
    print(f'  Total       : {k["mobilisation_cost"]:.2f}')
    by_machine = k['mobilisation_cost_by_machine']
    if isinstance(by_machine, dict):
        for m_id, cost in sorted(by_machine.items()):
            print(f'    {m_id}: {cost:.2f}')
    by_landing = k['mobilisation_cost_by_landing']
    if isinstance(by_landing, dict):
        for l_id, cost in sorted(by_landing.items()):
            print(f'    Landing {l_id}: {cost:.2f}')


=== tiny7 mobilisation cost ===
  Total       : 585.64

=== small21 mobilisation cost ===
  Total       : 2564.40


## Utilisation summary

Per-machine utilisation ratio shows how fully each machine is used
within its available shift hours.

In [8]:
from fhops.evaluation.playback.aggregates import machine_utilisation_summary

for kind in ['tiny7', 'small21']:
    s = schedules[kind]
    result = run_playback(s['problem'], s['assignments'])
    shift_df = shift_dataframe(result)
    util = machine_utilisation_summary(shift_df)
    print(f'\n=== {kind} machine utilisation ===')
    display(util.sort_values('utilisation_ratio', ascending=False))


=== tiny7 machine utilisation ===


,sample_id,machine_id,total_hours,available_hours,production_units,mobilisation_cost,blackout_conflicts,sequencing_violations,utilisation_ratio
0,0,H1,48.0,48.0,2415.615867,53.24,0,0,1.0
1,0,H2,48.0,48.0,1999.086885,53.24,0,0,1.0
2,0,H3,96.0,96.0,4414.702752,106.48,0,0,1.0
3,0,H4,96.0,96.0,2203.444318,106.48,0,0,1.0
4,0,H5,96.0,96.0,1932.052169,106.48,0,0,1.0
5,0,H6,24.0,24.0,279.206265,0.00,0,0,1.0
6,0,H7,96.0,96.0,3479.192358,106.48,0,0,1.0
7,0,H8,48.0,48.0,935.510394,53.24,0,0,1.0



=== small21 machine utilisation ===


,sample_id,machine_id,total_hours,available_hours,production_units,mobilisation_cost,blackout_conflicts,sequencing_violations,utilisation_ratio
0,0,H1,216.0,216.0,7246.065185,428.56,0,0,1.0
1,0,H2,216.0,216.0,8720.901635,214.88,0,0,1.0
2,0,H3,336.0,336.0,15966.966820,373.40,0,0,1.0
3,0,H4,336.0,336.0,7233.979586,373.40,0,0,1.0
4,0,H5,336.0,336.0,6893.022960,373.40,0,0,1.0
5,0,H6,144.0,144.0,1839.964274,160.44,0,0,1.0
6,0,H7,336.0,336.0,10640.772521,373.40,0,0,1.0
7,0,H8,192.0,192.0,5080.290336,213.44,0,0,1.0
8,0,H9,48.0,48.0,245.903963,53.48,0,0,1.0


## CLI playback & evaluation

Demonstrates the same pipeline via the installed/source `fhops` CLI. Outputs
are written to a temp directory and discarded at the end of the cell.

In [9]:
import json
import tempfile
from pathlib import Path

demo_tmp = tempfile.mkdtemp(prefix="fhops_cli_demo_")

for kind in ['tiny7', 'small21']:
    s = schedules[kind]
    pb = s['problem']
    csv_path = Path(demo_tmp) / f'{kind}_demo.csv'
    s['assignments'].to_csv(csv_path, index=False)

    print(f'\n=== {kind} CLI eval-playback ===')
    pb_result = run_fhops_cli(
        'eval-playback', str(repo_root / 'examples' / kind / 'scenario.yaml'),
        '--assignments', str(csv_path),
        '--shift-out', str(Path(demo_tmp) / f'{kind}_shift.csv'),
        '--day-out', str(Path(demo_tmp) / f'{kind}_day.csv'),
        '--summary-md', str(Path(demo_tmp) / f'{kind}_summary.md'),
    )
    print(pb_result['stdout'][:400])

    print(f'\n=== {kind} CLI evaluate ===')
    eval_result = run_fhops_cli(
        'evaluate', str(repo_root / 'examples' / kind / 'scenario.yaml'),
        '--assignments', str(csv_path),
        '--kpi-mode', 'basic',
    )
    print(eval_result['stdout'][:400])

# List what was produced
for f in sorted(Path(demo_tmp).iterdir()):
    print(f'  {f.name}  ({f.stat().st_size} bytes)')


=== tiny7 CLI eval-playback ===
                             Shift Playback Summary                             
┏━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ Machine ┃ Day ┃ Shift ┃    Prod ┃ Hours ┃ Idle ┃ Mobili… ┃[

=== tiny7 CLI evaluate ===
KPI Summary
Production
  total_production: 4414.703
  staged_production: 0.000
  remaining_work_total: 0.000
  completed_blocks: 2.000
  makespan_day: 7
  makespan_shift: S1
Mobilisation
  mobilisation_cost: 585.640
  mobilisation_cost_by_machine: H1=53.24, H2=53

=== small21 CLI eval-playback ===
                             Shift Playback Summary                             
┏━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ Machine ┃ Day ┃ Shift ┃    Prod ┃ Hours ┃ Idle ┃ Mobili… ┃[

=== small21 CLI evaluate ===
KPI Summary
Production
  total_production: 15966.967
  staged_production: 0.000
  remaining_work_total: 0.000
  completed_blocks: 6.000
  makespan_day: 17
  mak

## Summary

- Deterministic playback converts assignments into shift/day records.
- KPIs provide production, mobilisation, utilisation, and sequencing metrics.
- Next: `04_fhops_stochastic_what_if.ipynb` adds stochastic sampling
  (downtime/weather) to explore schedule robustness.